# 05 · Robustez e identificación causal

**Instrumento shift-share (Card, 2001; González y Ortega, 2013):** el flujo predicho asigna a cada provincia el crecimiento nacional de cada nacionalidad (excluida la propia provincia) según la distribución inicial de esa nacionalidad. Supuesto clave: las cuotas iniciales no están correlacionadas con shocks posteriores de demanda de vivienda (Goldsmith-Pinkham et al., 2020). Ver `docs/methodology.md`.

In [1]:
# Configuración: raíz del repositorio en el path
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

# Para comprobar el código sin datos reales: os.environ["INMIG_DEMO"] = "1"
from src import config
print("MODO DEMO (DATOS SINTÉTICOS)" if config.is_demo() else "Datos reales")

Datos reales


In [2]:
from src import data_cleaning as dc
panel = dc.load_panel()
print(panel.shape)
panel.head()

(1300, 31)


,provincia_cod,anio,precio_m2,precio_m2_t4,anio_padron,nacidos_extranjero,extranjeros,poblacion_total,poblacion,ln_precio,...,d_pct_extranjeros,flujo_extranjeros,flujo_nacidos_ext_l1,flujo_nacidos_ext_l2,flujo_extranjeros_l1,flujo_extranjeros_l2,pct_nacidos_ext_l1,pct_nacidos_ext_l2,ccaa,provincia
0,01,1997,1013.650,1039.5,1998,4603,2460,284595,284595,6.921313,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,País Vasco,Araba/Álava
1,01,1998,1151.875,1272.4,1999,4947,2801,285748,285748,7.049146,...,0.115848,0.119819,NaN,NaN,NaN,NaN,1.617386,NaN,País Vasco,Araba/Álava
2,01,1999,1392.325,1467.6,2000,6108,3818,286497,286497,7.238730,...,0.352415,0.355908,0.120874,NaN,0.119819,NaN,1.731246,1.617386,País Vasco,Araba/Álava
3,01,2000,1475.250,1554.6,2001,7850,5462,288793,288793,7.296583,...,0.558671,0.573828,0.406302,0.120874,0.355908,0.119819,2.131959,1.731246,País Vasco,Araba/Álava
4,01,2001,1653.375,1712.2,2002,10573,8031,291860,291860,7.410574,...,0.860342,0.889564,0.608034,0.406302,0.573828,0.355908,2.718210,2.131959,País Vasco,Araba/Álava


In [3]:
from src import analysis
nat = dc.load_nationalities()
ivr = analysis.stage_iv_robustness(panel, nat)
ivr.get('iv_table', 'IV no estimado: faltan datos por nacionalidad')

,(1) EF misma muestra,(2) 2SLS (flujo extranjeros),(3) 2SLS (flujo nacidos ext.),(4) Forma reducida
flujo_extranjeros,1.1040***,0.4428,,
flujo_extranjeros (ee),(0.2433),(0.4698),,
flujo_nacidos_ext,,,0.4712,
flujo_nacidos_ext (ee),,,(0.4987),
iv_shift_share,,,,0.3420
iv_shift_share (ee),,,,(0.3772)
Variable dependiente,dln_precio,dln_precio,dln_precio,dln_precio
Efectos fijos,provincia + año,provincia + año,provincia + año,provincia + año
Errores estándar,cluster provincia,cluster provincia,cluster provincia,cluster provincia
Observaciones,1216,1216,1216,1216


In [4]:
ivr['robustness'].round(4)

,modelo,variable,coef,se,p,ci_low,ci_high,nobs
0,Principal,flujo_nacidos_ext,1.2012,0.2588,0.0000,0.6934,1.7091,1216
1,Def. alternativa: extranjeros,flujo_extranjeros,1.1040,0.2433,0.0000,0.6266,1.5814,1216
2,Regresor: Δ proporción nacidos ext. (p.p.),d_pct_nacidos_ext,1.5475,0.3034,0.0000,0.9523,2.1428,1216
3,Precio 4.º trimestre (alineado con Padrón),flujo_nacidos_ext,1.1966,0.3008,0.0001,0.6064,1.7867,1216
4,Sin provincias atípicas,flujo_nacidos_ext,1.1913,0.3211,0.0002,0.5612,1.8213,1064
5,Sin outliers (p1-p99),flujo_nacidos_ext,1.3451,0.2574,0.0000,0.8401,1.8501,1195
6,Subperiodo: expansión (hasta 2007),flujo_nacidos_ext,0.6020,0.3175,0.0586,-0.0220,1.2259,506
7,Subperiodo: crisis (2008-2013),flujo_nacidos_ext,-0.3343,0.5891,0.5710,-1.4949,0.8263,295
8,Subperiodo: recuperación (2014 en adelante),flujo_nacidos_ext,1.5221,0.4694,0.0013,0.5989,2.4453,415
9,Niveles: ln precio ~ % nacidos ext.,pct_nacidos_ext,0.0088,0.0045,0.0515,-0.0001,0.0176,1276


In [5]:
ivr['loo'][['provincia','coef','ci_low','ci_high']].sort_values('coef').round(4)

,provincia,coef,ci_low,ci_high
42,Tarragona,1.1301,0.6298,1.6305
11,Castellón/Castelló,1.1449,0.6303,1.6595
29,Murcia,1.1468,0.6422,1.6514
3,Almería,1.1480,0.6215,1.6744
31,Ourense,1.1533,0.6523,1.6543
30,Navarra,1.1657,0.6601,1.6712
51,Melilla,1.1732,0.6421,1.7044
5,Badajoz,1.1758,0.6625,1.6891
16,Girona,1.1768,0.6594,1.6942
45,Valencia/València,1.1783,0.6706,1.6860


In [6]:
ivr['vif'], ivr['wooldridge']

(            variable  VIF
 0  flujo_nacidos_ext  1.0,
 {'rho': 0.6365406115129377,
  'F': 1762.2568504547185,
  'pvalue': 0.0,
  'nobs': 1164})

## Informe automático

El resumen se genera solo a partir de las estimaciones anteriores.

In [7]:
from src import analysis as a
ols = a.stage_ols(panel); pan = a.stage_panel(panel)
desc = a.stage_descriptive(panel)
from IPython.display import Markdown
Markdown(a.write_report(panel, desc, ols, pan, ivr))

# Resumen de resultados (generado automáticamente)

Muestra: 52 provincias, años 1997-2021. Observaciones en el modelo principal: 1216.
Controles incluidos: NINGUNO (solo efectos fijos de provincia y año; los controles económicos de Eurostat no estaban disponibles en esta ejecución).

## Modelo principal (efectos fijos de provincia y año, errores agrupados por provincia)

- Coeficiente del flujo de población nacida en el extranjero: 1.201 (IC 95 %: 0.693 a 1.709; p = 0.000).
- Interpretación: un aumento neto de residentes nacidos en el extranjero equivalente al 1 % de la población de la provincia se asocia con una variación de 1.20 puntos porcentuales en el crecimiento anual del precio de la vivienda, condicionada a los efectos fijos de provincia y año (y a los controles, si los hay).
- MCO agrupado sin efectos fijos: 5.485. La diferencia con el modelo principal indica qué parte de la correlación bruta se debe a diferencias permanentes entre provincias y al ciclo nacional común.
- Con la variación de la **proporción** de nacidos en el extranjero (p.p.) como regresor, que no depende del tamaño de la población: 1.548 (IC 95 %: 0.952 a 2.143; p = 0.000).
- Efecto acumulado (retardos 0-2): 1.145 (IC 95 %: 0.429 a 1.860; p = 0.002).
- H2 (efecto acumulado mayor que el contemporáneo): compatible con los datos.
- Especificación en niveles: +1 p.p. en el peso de la población nacida en el extranjero se asocia con un 0.88 % de variación del precio (IC 95 %: -0.01 % a 1.76 %).

## Variables instrumentales (shift-share)

- 2SLS: 0.471 (IC 95 %: -0.506 a 1.449; p = 0.345). Estadístico de primera etapa (Wald robusto a cluster, equivalente a F con un instrumento) = 47.78.
- El instrumento no parece débil según la regla F > 10, pero su validez depende de la exogeneidad de las cuotas iniciales (ver docs/methodology.md).
- Comparación: efectos fijos en la misma muestra con el flujo de extranjeros = 1.104; 2SLS con el flujo de extranjeros = 0.443. Forma reducida: 0.342 (IC 95 %: -0.398 a 1.082; p = 0.365).

## Robustez

- 7 de 11 especificaciones presentan p < 0.05. Rango de coeficientes de flujo: -0.334 a 1.548.
- Por especificación: Principal: 1.201 (p = 0.000); Def. alternativa: extranjeros: 1.104 (p = 0.000); Regresor: Δ proporción nacidos ext. (p.p.): 1.548 (p = 0.000); Precio 4.º trimestre (alineado con Padrón): 1.197 (p = 0.000); Sin provincias atípicas: 1.191 (p = 0.000); Sin outliers (p1-p99): 1.345 (p = 0.000); Subperiodo: expansión (hasta 2007): 0.602 (p = 0.059); Subperiodo: crisis (2008-2013): -0.334 (p = 0.571); Subperiodo: recuperación (2014 en adelante): 1.522 (p = 0.001); Niveles: ln precio ~ % nacidos ext.: 0.009 (p = 0.051); 2SLS shift-share (nacidos ext.): 0.471 (p = 0.345).
- Excluyendo provincias de una en una, el coeficiente varía entre 1.130 y 1.363.
- Test de Wooldridge de autocorrelación: p = 0.000.
- VIF máximo: 1.00.

## Advertencia sobre causalidad

Los coeficientes de MCO y efectos fijos son asociaciones condicionales. Solo la estimación IV aspira a una interpretación causal, y únicamente si se cumplen sus supuestos de identificación, que no pueden contrastarse de forma directa.
